# 강의 06 · 실습 4 — 패턴 4 수퍼바이저 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 총무팀 팀장은 부서장에게 낼 보고서 초안을 팀원에게 시킵니다.
- 초안은 조사 담당이 넣을 항목을 고르고, 작성 담당이 초안을 쓰고, 검토 담당이 고칠 점을 짚는 순서로 만들어집니다.
- 팀원은 이 순서를 코드의 화살표로 고정해 두었습니다. 그래서 순서를 바꾸거나 한 담당을 건너뛰려면 코드를 고쳐야 하고, 담당이 끝난 뒤 다음에 누구를 부를지를 실행 중에 정할 길이 없습니다.
- 팀장이 원하는 것은 보고 요청만 주면 팀장 역할이 담당들을 차례로 시키고 다 끝나면 스스로 멈추는 흐름입니다.

## 2. 문제와 목표

- **문제**: 다음 에이전트를 정하는 판단이 코드의 엣지에만 있습니다. 실행 중에 순서를 정하거나 바꿀 수 없고, 에이전트가 끝난 뒤 누구를 부를지를 보고 판단할 단계가 없습니다.
- **목표**: 보고 요청을 넣으면 supervisor 노드가 끝난 에이전트 목록을 보고 다음 에이전트를 구조화 출력으로 정해 그 에이전트로 이동시키고, 에이전트가 끝나면 다시 supervisor로 돌아오며, 셋이 다 끝나면 supervisor가 스스로 종료를 고르는 처리 흐름을 만듭니다.
    - supervisor 노드: 끝난 에이전트 목록을 보고 다음 담당을 구조화 출력 `Assign`(research·write·review·done)으로 정합니다.
    - 에이전트 셋: research(항목 뽑기), write(초안 쓰기), review(검토) — 각각 노드 하나짜리 서브그래프.
    - 종료: supervisor가 done을 고르거나 진입 횟수가 상한 4에 닿을 때.
- **목표 달성 여부의 판정 기준**: 보고 요청을 입력했을 때, supervisor가 research·write·review 에이전트를 한 번씩 차례로 부르고 네 번째 진입에서 스스로 종료를 고르는 것과, 최종 작업 기록에 세 에이전트의 결과가 그 순서대로 남는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex04_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 배정 규격을 정의합니다.**
    - 보고 요청(`request`), 작업 기록(`log`), 초안(`draft`), supervisor 진입 횟수(`step`) 키 네 개를 가지는 상태를 선언합니다.
    - `log`에는 리듀서를 걸지 않습니다.
    - 배정 규격 `Assign`은 `who` 필드 하나를 가지며, 값은 research·write·review·done 넷 중 하나로만 제한합니다.
    - 진입 횟수 상한은 4로 둡니다.
2. **supervisor 노드를 만듭니다.**
    - 진입 횟수가 상한에 닿으면 END로 가는 `Command`를 돌려줍니다.
    - 아니면 작업 기록에서 끝난 에이전트 이름을 뽑아, `Assign` 규격을 건 모델에 「research, write, review 순서로 한 번씩 시키고 셋이 다 끝났으면 done」이라는 지침과 함께 넣어 다음 에이전트를 받습니다.
    - 받은 값이 done이면 END로, 아니면 그 에이전트로 가면서 `step`을 1 올리는 `Command(goto=..., update=...)`를 돌려줍니다.
3. **에이전트 서브그래프 세 개를 만듭니다.**
    - research는 보고서에 넣을 항목 2개를 한 줄로 뽑아 `log`에 「research: 항목」 줄을 덧붙입니다.
    - write는 요청과 작업 기록을 참고해 초안을 두 문장으로 써서 `draft`에 넣고 `log`에 「write: 초안 작성」 줄을 덧붙입니다.
    - review는 초안에서 고칠 점 하나를 한 문장으로 말해 `log`에 「review: 의견」 줄을 덧붙입니다.
    - 세 에이전트는 각각 노드 하나짜리 서브그래프로 만들고, 부모와 같은 `ReportState`를 씁니다.
    - `log`는 기존 목록에 새 줄을 더한 전체를 돌려줍니다.
4. **그래프에 노드를 등록합니다.**
    - supervisor 함수와, 컴파일된 서브그래프 세 개를 research·write·review라는 이름으로 등록합니다.
    - 에이전트 이름이 그대로 노드 이름이고, supervisor가 고르는 값과 같아야 합니다.
5. **엣지를 연결합니다.**
    - START에서 supervisor로 가는 고정 엣지를 추가합니다.
    - research·write·review 뒤에는 각각 supervisor로 돌아가는 고정 엣지를 추가합니다.
    - supervisor에서 에이전트로 나가는 엣지와 조건부 엣지는 추가하지 않습니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 보고 요청과 빈 작업 기록, 빈 초안, 진입 횟수 0을 넣어 실행한 뒤, 최종 진입 횟수와 작업 기록 전체와 초안을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키와 배정 규격을 선언합니다 | `class ReportState(TypedDict)`, `class Assign(BaseModel)` | 1 |
| ② 노드 함수 정의 | 에이전트를 정하며 이동하는 supervisor와 에이전트 서브그래프를 만듭니다 | `Command(goto=..., update=...)`, `StateGraph(ReportState).compile()` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 supervisor와 서브그래프 셋을 등록합니다 | `StateGraph(ReportState)`, `add_node` | 4 |
| ④ 엣지 연결 | 에이전트에서 수퍼바이저로 돌아오는 순서만 고정합니다 | `add_edge` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 요청을 넣어 실행합니다 | `compile()`, `invoke()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [1]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기, 모델 준비)을 작성합니다.
import os

from dotenv import load_dotenv, find_dotenv
from typing import Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from pydantic import BaseModel, Field

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

모델 준비를 마쳤습니다.


### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `step`은 supervisor에 들어온 횟수이며 상한 판정의 근거입니다. `log`에는 리듀서를 걸지 않습니다. 부모와 같은 스키마를 쓰는 서브그래프는 끝날 때 `log` 전체를 돌려주므로, 리듀서까지 걸면 같은 항목이 두 번 쌓입니다. 배정 규격 `Assign`은 다음 에이전트의 이름만 담고, 에이전트 후보와 종료 신호 done이 같은 목록에 들어갑니다.

In [2]:
# 여기에 단계 ①(상태 정의와 배정 규격 Assign 정의, 상한 상수)을 작성합니다.
class ReportState(TypedDict):
    request: str   # 부서장이 시킨 일
    log: list      # 누가 무엇을 했는지 (리듀서 없음: 서브그래프가 전체를 돌려준다)
    draft: str     # 작성 결과
    step: int      # supervisor 진입 횟수 (상한 판정용)

class Assign(BaseModel):
    who: Literal["research", "write", "review", "done"] = Field(
        description="다음에 일할 에이전트. 더 시킬 일이 없으면 done")

MAX_STEPS = 4

print("상태의 키:", list(ReportState.__annotations__))
print("배정 값의 범위:", Assign.model_json_schema()["properties"]["who"]["enum"])

상태의 키: ['request', 'log', 'draft', 'step']
배정 값의 범위: ['research', 'write', 'review', 'done']


### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- supervisor는 상태를 바꾸면서 동시에 갈 곳도 정합니다. 그래서 딕셔너리가 아니라 `Command`를 돌려줍니다. `goto`가 이동, `update`가 상태 갱신입니다. 조건부 엣지는 갈 곳만 정하고 상태를 바꾸지 않으므로 여기서는 쓰지 않습니다.
- supervisor는 끝난 에이전트 목록을 프롬프트에 실어 보냅니다. 서브에이전트는 지난 대화를 기억하지 않으므로, 필요한 맥락은 수퍼바이저가 명시해서 실어 보냅니다.
- 에이전트 세 개는 노드 하나짜리 서브그래프입니다. `StateGraph(ReportState)`로 부모와 같은 스키마를 쓰고, 컴파일한 결과를 부모 그래프의 노드로 씁니다. 서브그래프 안의 노드 이름(collect·draft·check)은 부모의 노드 이름과 달라도 됩니다.
- write는 앞 에이전트가 남긴 `log`를 초안 재료로 읽습니다. 순서를 수퍼바이저가 정하는 의미가 여기서 생깁니다.

In [8]:
assigner = llm.with_structured_output(Assign)


def supervisor(state: ReportState) -> Command:
    """끝난 에이전트 목록을 보고 다음 에이전트를 정하고, step을 올리면서 그 에이전트로 이동한다."""
    if state["step"] >= MAX_STEPS:
        print("  [supervisor] 상한 도달 -> 종료")
        return Command(goto=END, update={"log": state["log"] + ["supervisor: 상한 종료"]})
    done = [x.split(":")[0] for x in state["log"]]
    res = assigner.invoke([
        SystemMessage("보고서 작업을 research(자료 정리) -> write(초안) -> review(검수) "
                      "순서로 한 번씩만 시킨다. 셋이 다 끝났으면 done."),
        HumanMessage(f"요청: {state['request']}\n끝난 에이전트: {done}"),
    ])
    print(f"  [supervisor] 진입 step={state['step']} -> who={res.who!r}")
    nxt = END if res.who == "done" else res.who
    return Command(goto=nxt, update={"step": state["step"] + 1})


def collect(state: ReportState) -> dict:
    """조사 에이전트: 보고서에 넣을 항목 2개를 뽑는다."""
    print("      [sub:research/collect] 자료 항목 뽑기")
    res = llm.invoke([
        SystemMessage("조사 에이전트다. 보고서에 넣을 항목 2개를 한 줄로 나열한다."),
        HumanMessage(state["request"]),
    ])
    return {"log": state["log"] + [f"research: {res.content.strip()[:36]}"]}


def build_research_sub():
    sub = StateGraph(ReportState)      # 부모와 같은 스키마
    sub.add_node("collect", collect)
    sub.add_edge(START, "collect")
    sub.add_edge("collect", END)
    return sub.compile()


def write_draft(state: ReportState) -> dict:
    """작성 에이전트: 앞 에이전트의 기록을 참고해 초안을 두 문장으로 쓴다."""
    print("      [sub:write/draft] 초안 쓰기")
    res = llm.invoke([
        SystemMessage("작성 에이전트다. 받은 자료를 참고해 보고서 초안을 두 문장으로 쓴다."),
        HumanMessage(f"{state['request']}\n자료: {state['log']}"),
    ])
    return {"draft": res.content.strip(), "log": state["log"] + ["write: 초안 작성"]}


def build_write_sub():
    sub = StateGraph(ReportState)
    sub.add_node("draft", write_draft)
    sub.add_edge(START, "draft")
    sub.add_edge("draft", END)
    return sub.compile()


def review(state: ReportState) -> dict:
    """검토 에이전트: 초안에서 고칠 점 하나를 짚는다."""
    print("      [sub:review/check] 검수")
    res = llm.invoke([
        SystemMessage("검토 에이전트다. 초안에서 고칠 점 하나만 한 문장으로 말한다."),
        HumanMessage(state["draft"]),
    ])
    return {"log": state["log"] + [f"review: {res.content.strip()[:36]}"]}


def build_review_sub():
    sub = StateGraph(ReportState)
    sub.add_node("check", review)
    sub.add_edge(START, "check")
    sub.add_edge("check", END)
    return sub.compile()

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 노드를 등록합니다. 에이전트 세 개는 함수가 아니라 컴파일된 서브그래프를 그대로 노드로 등록합니다. 서브그래프가 부모와 같은 상태 스키마를 쓰므로 별도 변환 없이 얹힙니다. 이 구조에서 서브에이전트는 서브그래프로 구현됩니다.

In [10]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.
g = StateGraph(ReportState)
g.add_node("supervisor", supervisor)
g.add_node("research", build_research_sub())   # 컴파일된 서브그래프를 노드로 직접 얹는다
g.add_node("write", build_write_sub())
g.add_node("review", build_review_sub())

print("등록한 노드:", list(g.nodes))

등록한 노드: ['supervisor', 'research', 'write', 'review']


### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. START에서 supervisor로 가는 엣지와, 에이전트 세 개에서 supervisor로 돌아오는 엣지를 추가합니다. supervisor에서 에이전트로 나가는 엣지는 추가하지 않습니다. supervisor가 돌려주는 `Command(goto=...)`가 다음 노드를 정하기 때문입니다. 다음 에이전트를 정하는 판단이 supervisor 한 곳에만 있어 중앙 집중형이 성립합니다.

In [11]:
# 여기에 단계 ④(엣지 연결)를 작성합니다.
g.add_edge(START, "supervisor")
for name in ("research", "write", "review"):
    g.add_edge(name, "supervisor")   # 에이전트는 끝나면 항상 수퍼바이저로 돌아온다

print("고정 엣지 수:", len(g.edges))

고정 엣지 수: 4


### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 요청과 빈 기록을 넣으면 최종 상태가 돌아옵니다. 실행 중 supervisor와 에이전트가 출력하는 진입 줄로 수퍼바이저가 에이전트를 부르는 순서를 봅니다. 아래에서는 회의실 예약 시스템 개편 결과 보고를 요청합니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.
graph = g.compile()

REQUEST = "회의실 예약 시스템 개편 결과 보고"

print(f"=== 요청: {REQUEST} ===")
out = graph.invoke({"request": REQUEST, "log": [], "draft": "", "step": 0})

print()
print(f"[최종 상태] step={out['step']} 작업 기록 {len(out['log'])}줄")
for line in out["log"]:
    print(f"  - {line}")
print("--- 초안 ---")
print(out["draft"])

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. `[supervisor] 진입` 줄이 네 번 출력되고, `who` 값이 research, write, review, done 순서입니다. 각 진입 줄 뒤에 그 에이전트의 `[sub:...]` 줄이 하나씩 붙습니다.
2. 에이전트가 끝날 때마다 supervisor로 돌아옵니다. 에이전트의 줄 다음에는 항상 supervisor의 진입 줄이 옵니다.
3. 최종 상태의 `step`은 4이고, 작업 기록은 research·write·review 세 줄이 순서대로 남아 있습니다. 같은 줄이 두 번 쌓이지 않습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. 작업 기록의 줄이 두 번씩 쌓이면 단계 ①에서 `log`에 리듀서를 걸었는지, 에이전트가 끝난 뒤 멈추면 단계 ④에서 에이전트에서 supervisor로 돌아오는 엣지를 다시 봅니다.